# INT8 Quantization Benchmark

In this notebook we are going to investigate the performance of INT8 Model of phi2, comparing in benchmark the models.

The objetives are:

    - INT8 reduces the memory?
    - Is quicker than baseline?
    - Losses quality the response?
    

## Imports

In [14]:
import time
import gc
import torch
import psutil
import pandas as pd

from transformers import(
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

## Configuration

In [15]:
MODEL_ID = "microsoft/phi-2"

PROMPT = """
You are an AI expert.

Explain quantization in large language models
for a beginner.

Use simple language and avoid code.
"""

MAX_NEW_TOKENS = 150

## Device

As bitsandbytes and Apple Silicon have a limited technical support we have to use the CPU. The model quantizied INT8 could not download on MPS and automatically on CPU.

In [16]:
device = torch.device("cpu")

print("Using CPU for INT8 inference")

Using CPU for INT8 inference


## RAM Helper

In [17]:
def get_ram_usage_gb():
    process = psutil.Process()
    memory_info = process.memory_info()
    return memory_info.rss / (1024 ** 3)

## Cleanup header

In [18]:
def cleanup():
    gc.collect()

    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

## Load Tokenizer

In [19]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

## INT8 Quantization Config

This means that we download the weights quantized in INT8, instead of saving FP16 (16 Bits), it saves INT8(8 Bits). The weights occupy less memory.

In [20]:
quant_config = BitsAndBytesConfig(
    load_in_8bit=True
)

## Load Quantized model

In this part we load the INT8 Model with the quant_config we established. Also we establish low_cpu_mem_usage which is util because reduces temporal RAM and memory spikes.

In [21]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    low_cpu_mem_usage=True
)

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

## Tokenization

In [22]:
inputs = tokenizer(
    PROMPT,
    return_tensors="pt"
)

inputs = {k: v.to(device) for k, v in inputs.items()}

## Benchmark

In [23]:
initial_ram = get_ram_usage_gb()

start_time = time.time()

outputs = model.generate(
    **inputs,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=True,
    temperature=0.7,
    eos_token_id=tokenizer.eos_token_id
)

end_time = time.time()

final_ram = get_ram_usage_gb()

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


## Decode

In [24]:
generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)

print(generated_text)


You are an AI expert.

Explain quantization in large language models
for a beginner.

Use simple language and avoid code.

Solution:

A large language model is a type of model that is built to understand and generate text. It is called "large" because it has a really big number of parameters or weights that it uses to make predictions. These parameters are like the rules or instructions that tell the model what to do when it sees or hears different words.

But since a large language model has so many parameters, it uses a lot of memory and processing power. So, we have to make a special version of it called a "quantized" version. This means we reduce the number of parameters by using a different way to represent the words.

The new way is called "quantization" and it is like putting the words into different


## Metrics

In [25]:
total_time = end_time - start_time

generated_tokens = outputs.shape[1]

tokens_per_second = generated_tokens / total_time

print(f"Inference Time: {total_time:.2f} sec")
print(f"Tokens/sec: {tokens_per_second:.2f}")
print(f"RAM Usage: {(final_ram - initial_ram):.2f} GB")

Inference Time: 8.91 sec
Tokens/sec: 20.43
RAM Usage: 0.40 GB


## Save results

In [26]:
results = {
    "model": "Phi-2",
    "precision": "INT8",
    "inference_time": total_time,
    "tokens_per_second": tokens_per_second,
    "ram_usage_gb": final_ram - initial_ram
}

df = pd.DataFrame([results])

df.to_csv(
    "../results/phi2_int8.csv",
    index=False
)

print(df)

   model precision  inference_time  tokens_per_second  ram_usage_gb
0  Phi-2      INT8        8.907587          20.432021      0.397858


## Cleanup

In [27]:
del model 

cleanup()